In [ ]:
import argparse
import copy
import json
import os
from dataclasses import dataclass
from datetime import datetime
from numba import njit
# --- third-party ---
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.linalg import solve_triangular
from scipy.special import gammaln
from sklearn.covariance import GraphicalLassoCV
from sklearn.metrics import f1_score, precision_score, recall_score


In [ ]:
@njit(cache=True)
def _cholesky_update_numba(L, v):
    L_new = L.copy()
    v_cur = v.copy()
    N = len(v)
    for k in range(N):
        L_kk = L_new[k, k]
        v_k = v_cur[k]
        r = np.sqrt(L_kk ** 2 + v_k ** 2)
        c = L_kk / r
        s = v_k / r
        L_new[k, k] = r
        if k < N - 1:
            L_k_sub = L_new[k + 1:, k].copy()
            v_sub = v_cur[k + 1:].copy()
            L_new[k + 1:, k] = c * L_k_sub + s * v_sub
            v_cur[k + 1:] = -s * L_k_sub + c * v_sub
    return L_new

@njit(cache=True)
def _cholesky_downdate_numba(L, v):
    L_new = L.copy()
    v_cur = v.copy()
    N = len(v)
    for k in range(N):
        L_kk = L_new[k, k]
        v_k = v_cur[k]
        diff = L_kk ** 2 - v_k ** 2
        if diff <= 0:
            raise ValueError("non-PD downdate")
        r = np.sqrt(diff)
        c = L_kk / r
        s = v_k / r
        L_new[k, k] = r
        if k < N - 1:
            L_k_sub = L_new[k + 1:, k].copy()
            v_sub = v_cur[k + 1:].copy()
            L_new[k + 1:, k] = c * L_k_sub - s * v_sub
            v_cur[k + 1:] = -s * L_k_sub + c * v_sub
    return L_new

# class for model

In [ ]:
class SparseFactorizedPrior:
    """Manages the factorized precision matrix prior and matrix operations.

    K = eps * I + B @ B.T, where each column b_r of B has exactly two
    nonzero entries at positions z_r = (i_r, j_r), with values
    (alpha_r, beta_r) ~ N(0,1) iid.
    """

    def __init__(
        self, N: int, R: int = 0, eps: float = 1e-3,
        lam_poisson: float | None = None, target_d: float = 3.0,
    ):
        self.N = N
        self.eps = eps

        if lam_poisson is None:
            self.lam_poisson = (N * target_d) / 2.0
        else:
            self.lam_poisson = lam_poisson

        # Candidate edge pairs (i, j) with i < j
        self.all_pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]
        self.E_total = len(self.all_pairs)  # N*(N-1)/2

        self._R_init = R

        if R > 0:
            selected_indices = np.random.choice(self.E_total, size=R, replace=True)
            self.z = np.array(self.all_pairs)[selected_indices]
            theta_init = np.random.randn(R, 2)
        else:
            self.z = np.empty((0, 2), dtype=int)
            theta_init = np.empty((0, 2))

        self.set_theta(theta_init)

    @property
    def R(self) -> int:
        """Current number of active columns."""
        return len(self.z)

    def set_theta(self, new_theta: np.ndarray):
        """Updates theta and recomputes B, K, and the Cholesky factor L."""
        self.theta = new_theta
        self._update_B_and_K()

    def update_factor(
        self, r: int, new_pair: tuple[int, int], theta_new_r: np.ndarray
    ) -> bool:
        """Rank-1 downdate (remove old column r) followed by rank-1 update
        (add new column r), used by both the swap move (new_pair != old
        pair) and single-column MALA (new_pair == old pair, weights only).
        Returns False (and leaves state untouched by caller's responsibility
        to roll back) if the downdate would be non-PD.
        """
        old_pair = tuple(self.z[r])
        theta_old_r = self.theta[r].copy()

        v_old = np.zeros(self.N)
        v_old[old_pair[0]], v_old[old_pair[1]] = theta_old_r[0], theta_old_r[1]

        v_new = np.zeros(self.N)
        v_new[new_pair[0]], v_new[new_pair[1]] = theta_new_r[0], theta_new_r[1]

        try:
            K_mid = self.K - np.outer(v_old, v_old)
            L_mid = self._cholesky_downdate(self.L, v_old)

            self.K = K_mid + np.outer(v_new, v_new)
            self.L = self._cholesky_update(L_mid, v_new)
        except ValueError:
            return False

        old_i, old_j = self.z[r]
        self.B[old_i, r] = 0.0
        self.B[old_j, r] = 0.0

        self.z[r] = new_pair
        self.theta[r] = theta_new_r

        self.B[new_pair[0], r] = theta_new_r[0]
        self.B[new_pair[1], r] = theta_new_r[1]

        return True

    def _update_B_and_K(self):
        """Full recompute of B, K, and Cholesky factor L from scratch.
        Cost: O(N^3) (dominated by the Cholesky factorization). Used at
        initialization and whenever a full-state rebuild is required.
        """
        self.B = np.zeros((self.N, self.R))
        for r in range(self.R):
            i, j = self.z[r]
            alpha, beta = self.theta[r]
            self.B[i, r] = alpha
            self.B[j, r] = beta

        self.K = self.eps * np.eye(self.N) + self.B @ self.B.T
        self.L = np.linalg.cholesky(self.K)

    def _rebuild_B(self):
        """Rebuilds B from (z, theta) without touching K/L (used after a
        rollback where K/L have already been restored separately)."""
        self.B = np.zeros((self.N, self.R))
        if self.R > 0:
            for r in range(self.R):
                i, j = self.z[r]
                self.B[i, r], self.B[j, r] = self.theta[r]

    def log_prior(self) -> float:
        """Full joint log-prior log P(z, theta), including the Poisson(lam)
        prior on R. NOTE: not called by the fixed-R swap move's acceptance
        ratio (the uniform prior on z contributes a ratio of 1 when R is
        fixed and z is drawn uniformly over pairs, so it cancels; only the
        Gaussian prior on theta matters there -- see Section 2). This
        method remains useful for the Geweke test and for reference.
        """
        R = self.R
        log_p_R = R * np.log(self.lam_poisson) - self.lam_poisson - gammaln(R + 1)

        if R == 0:
            return log_p_R

        log_P_z = -R * np.log(self.E_total)
        log_P_theta = -R * np.log(2 * np.pi) - 0.5 * np.sum(self.theta ** 2)

        return log_p_R + log_P_z + log_P_theta

    def sample_prior(self):
        """Samples (z, theta) from the full Poisson(lam)-R generative prior.
        Used for generating ground truth (Section 5), NOT for fixed-R
        inference (Section 3) or the fixed-R Geweke test (Section 7).
        """
        R_sampled = np.random.poisson(self.lam_poisson)

        if R_sampled > 0:
            selected_indices = np.random.choice(self.E_total, size=R_sampled, replace=True)
            self.z = np.array(self.all_pairs)[selected_indices]
            theta_init = np.random.randn(R_sampled, 2)
        else:
            self.z = np.empty((0, 2), dtype=int)
            theta_init = np.empty((0, 2))

        self.set_theta(theta_init)
        return self.K

    @staticmethod
    def _cholesky_update(L: np.ndarray, v: np.ndarray) -> np.ndarray:
        """Rank-1 Cholesky update. Delegates to numba-jitted implementation
        (identical arithmetic/order; only faster). Cost: O(N^2)."""
        return _cholesky_update_numba(L, v)

    @staticmethod
    def _cholesky_downdate(L: np.ndarray, v: np.ndarray) -> np.ndarray:
        """Rank-1 Cholesky downdate. Delegates to numba-jitted implementation.
        Raises ValueError if the result would not be PD."""
        try:
            return _cholesky_downdate_numba(L, v)
        except Exception:
            raise ValueError("Downdate results in a non-positive-definite matrix")


class PosteriorEvaluator:
    """Evaluates log-likelihood, log-posterior, and the gradient of the
    log-posterior w.r.t. theta, for a fixed support z."""

    def __init__(self, model: SparseFactorizedPrior, X: np.ndarray):
        self.model = model
        self.X = X
        self.N, self.T = X.shape
        self.S = (X @ X.T) / self.T

    def log_likelihood(self) -> float:
        """log p(X | K) = (T/2) log det K - (T/2) tr(K S) + const."""
        L = self.model.L
        logdet_K = 2.0 * np.sum(np.log(np.diag(L)))
        K = self.model.K
        tr_KS = np.sum(K * self.S)
        return 0.5 * self.T * logdet_K - 0.5 * self.T * tr_KS

    def log_posterior(self) -> float:
        return self.log_likelihood() + self.model.log_prior()

    def grad_theta(self):
        L = self.model.L
        B = self.model.B
        z = self.model.z
        theta = self.model.theta
        T = self.T
        R = self.model.R
        S = self.S

        grad = np.zeros_like(theta)
        if R == 0:
            return grad

        # K^{-1}B を全列まとめて解く(scipyのsolve_triangular呼び出しを R*2回 -> 2回に削減)
        Y = solve_triangular(L, B, lower=True)
        Kinv_B = solve_triangular(L, Y, lower=True, trans='T')

        # S@b_r は非ゼロ位置(i,j)から直接gatherして計算(密行列積 S@B は避ける -> O(RN)のまま)
        S_col_i = S[:, z[:, 0]]  # (N, R)
        S_col_j = S[:, z[:, 1]]  # (N, R)
        SB = theta[:, 0][None, :] * S_col_i + theta[:, 1][None, :] * S_col_j  # (N, R)

        M = Kinv_B - SB

        idx_r = np.arange(R)
        grad[:, 0] = T * M[z[:, 0], idx_r] - theta[:, 0]
        grad[:, 1] = T * M[z[:, 1], idx_r] - theta[:, 1]

        return grad

In [ ]:
def log_phi_2d(u: np.ndarray, sigma) -> float:
    """Computes log density of a 2D standard Gaussian log phi(u)."""
    return -0.5 * np.sum(u**2) / (sigma ** 2) - np.log(2.0 * np.pi * (sigma ** 2))


def birth_death_step(
    model: SparseFactorizedPrior, evaluator: PosteriorEvaluator,
) -> tuple[bool, float]:
    """Executes a Reversible Jump Birth-Death MH step matching your original logic."""
    M = model.E_total
    all_pairs = model.all_pairs
    n_current = len(model.z)
    sigma_prop = 1 / np.sqrt(evaluator.T)

    do_birth = (np.random.rand() < 0.5)

    # Boundary check
    if (not do_birth) and n_current == 0:
        return False, -np.inf

    log_p_before = evaluator.log_posterior()
    z_old, theta_old = model.z.copy(), model.theta.copy()
    K_old, L_old = model.K.copy(), model.L.copy()

    if do_birth:
      idx = np.random.randint(M)
      new_pair = all_pairs[idx]
      u = sigma_prop * np.random.randn(2)

      if n_current > 0:
        z_new = np.vstack([model.z, new_pair])
        theta_new = np.vstack([model.theta, u])
      else:
        z_new = np.array([new_pair], dtype = int)
        theta_new = u.reshape(1, 2)

      model.z = z_new
      model.theta = theta_new

      i, j = new_pair
      v = np.zeros(model.N)
      v[i], v[j] = u[0], u[1]

      model.K = K_old + np.outer(v, v)
      model.L = model._cholesky_update(L_old, v)
      model.B = np.zeros((model.N, model.R))
      for r in range(model.R):
        model.B[model.z[r, 0], r] = model.theta[r, 0]
        model.B[model.z[r, 1], r] = model.theta[r, 1]

      log_p_after = evaluator.log_posterior()

      # log q_ratio = log(q_backward / q_forward)
      log_ratio = (
          (log_p_after - log_p_before)
          - log_phi_2d(u, sigma = sigma_prop)
          + np.log(M)
      )

    else:
        r = np.random.randint(n_current)
        u = model.theta[r].copy()
        removed_pair = model.z[r]

        z_new = np.delete(model.z, r, axis=0)
        theta_new = np.delete(model.theta, r, axis=0)

        model.z = z_new
        model.theta = theta_new

        i, j = removed_pair
        v = np.zeros(model.N)
        v[i], v[j] = u[0], u[1]

        try:
          model.K = K_old -np.outer(v, v)
          model.L = model._cholesky_downdate(L_old, v)
        except ValueError:
            model.z, model.theta, model.K, model.L = z_old, theta_old, K_old, L_old
            model._update_B_and_K()
            return False, -np.inf

        model.B = np.zeros((model.N, model.R))
        for r_idx in range(model.R):
            model.B[model.z[r_idx, 0], r_idx] = model.theta[r_idx, 0]
            model.B[model.z[r_idx, 1], r_idx] = model.theta[r_idx, 1]

        log_p_after = evaluator.log_posterior()

        # log q_ratio = log(q_backward / q_forward)
        log_ratio = (
            (log_p_after - log_p_before)
            + log_phi_2d(u, sigma = sigma_prop)
            - np.log(M)
        )

    # Metropolis-Hastings Accept / Reject
    if np.log(np.random.rand()) < log_ratio:
        return True, log_ratio
    else:
        model.z, model.theta, model.K, model.L = z_old, theta_old, K_old, L_old
        model._rebuild_B()
        return False, log_ratio

#MALA

In [ ]:
@dataclass
class MALAConfig:
    c_init: float = 0.1
    alpha: float = -1 / 3
    target_accept: float = 0.574
    gamma: float = 0.2


class AdaptiveMALAStepper:
    def __init__(self, config: MALAConfig = MALAConfig()):
        self.c = config.c_init
        self.alpha = config.alpha
        self.target_accept = config.target_accept
        self.gamma = config.gamma
        self.is_frozen = False

    def get_h(self, N: int, R: int, T: int) -> float:
        """d = R*N (joint embedding dimension of all R columns)."""
        if R == 0:
            return 0.01
        d = R * N
        return float(self.c * (d ** self.alpha) / max(1, T))

    def adapt(self, recent_accept_rate: float):
        if self.is_frozen:
            return
        log_c = np.log(self.c) + self.gamma * (recent_accept_rate - self.target_accept)
        self.c = float(np.exp(log_c))

    def freeze(self):
        self.is_frozen = True


def propose_theta_ula(
    model, evaluator, h: float
) -> tuple[np.ndarray, np.ndarray]:
    """Proposes new theta (all R columns jointly) via ULA:
    theta' = theta + (h/2)*grad_theta + sqrt(h)*eta, eta ~ N(0, I).
    """
    theta = model.theta
    grad_at_theta = evaluator.grad_theta()

    if model.R == 0:
        return np.empty((0, 2)), np.empty((0, 2))

    eta = np.random.randn(*theta.shape)
    theta_new = theta + (h / 2.0) * grad_at_theta + np.sqrt(h) * eta
    return theta_new, grad_at_theta


def log_q_density(
    theta_to: np.ndarray, theta_from: np.ndarray,
    grad_at_from: np.ndarray, h: float,
) -> float:
    """log q(theta_to | theta_from) = N(theta_to; theta_from + (h/2)*grad, hI)."""
    if theta_to.size == 0:
        return 0.0
    mean = theta_from + (h / 2.0) * grad_at_from
    diff = theta_to - mean
    return -0.5 / h * np.sum(diff ** 2)


def mala_step(model, evaluator, stepper: AdaptiveMALAStepper) -> tuple[bool, float, float]:
    """Full-theta (all R columns jointly) MALA step. Requires a full
    Cholesky rebuild via model.set_theta on every proposal -- O(N^3).

    Returns: (accepted, log_accept_ratio, h)
    """
    T = evaluator.T
    R = model.R
    N = model.N
    if R == 0:
        return True, 0.0, stepper.get_h(N, 0, T)

    h = stepper.get_h(N, R, T)

    theta_old = model.theta.copy()
    log_p_old = evaluator.log_posterior()

    theta_new, grad_old = propose_theta_ula(model, evaluator, h)

    model.set_theta(theta_new)
    log_p_new = evaluator.log_posterior()
    grad_new = evaluator.grad_theta()

    log_q_forward = log_q_density(theta_new, theta_old, grad_old, h)
    log_q_backward = log_q_density(theta_old, theta_new, grad_new, h)

    log_accept_ratio = (log_p_new - log_p_old) + (log_q_backward - log_q_forward)

    if np.log(np.random.rand()) < log_accept_ratio:
        return True, log_accept_ratio, h
    else:
        model.set_theta(theta_old)
        return False, log_accept_ratio, h

# swap mpve

In [ ]:
def build_cov_bias_weights(S: np.ndarray, all_pairs: list[tuple[int, int]]) -> np.ndarray:
    weights = np.array([abs(S[i, j]) for (i, j) in all_pairs])
    return weights / weights.sum()


def swap_move_step_cov_local(model: SparseFactorizedPrior,
                              evaluator: PosteriorEvaluator,
                              pair_probs: np.ndarray,
                              pair_to_idx: dict[tuple[int, int], int],
                              sigma: float = 0.01) -> tuple[bool, float]:
    """
    R fixed swap move。
    z: coviarance
    theta: theta_new = theta_old + sigma * eta
    """
    R = model.R
    if R == 0:
      return False, -np.inf

    all_pairs = model.all_pairs
    z_old = model.z.copy()
    theta_old = model.theta.copy()
    K_old, L_old = model.K.copy(), model.L.copy()

    log_lik_before = evaluator.log_likelihood()

    r = np.random.randint(R)
    old_pair = tuple(model.z[r])
    old_pair_idx = pair_to_idx[old_pair]

    new_pair_idx = np.random.choice(len(all_pairs), p=pair_probs)
    new_pair = tuple(all_pairs[new_pair_idx])

    theta_old_r = model.theta[r].copy()
    theta_new_r = theta_old_r + sigma * np.random.randn(2)

    success = model.update_factor(r, new_pair, theta_new_r)
    if not success:
        model.z, model.theta, model.K, model.L = z_old, theta_old, K_old, L_old
        model._rebuild_B()
        return False, -np.inf

    log_lik_after = evaluator.log_likelihood()

    log_q_correction = np.log(pair_probs[old_pair_idx]) - np.log(pair_probs[new_pair_idx])

    # prior ratio (theta)
    log_prior_ratio = -0.5 * (np.sum(theta_new_r**2) - np.sum(theta_old_r**2))

    log_ratio = (log_lik_after - log_lik_before) + log_q_correction + log_prior_ratio

    if np.log(np.random.rand()) < log_ratio:
        return True, log_ratio
    else:
        model.z, model.theta, model.K, model.L = z_old, theta_old, K_old, L_old
        model._rebuild_B()
        return False, log_ratio

# run mcmc function

In [ ]:
def run_mcmc(
    model,
    evaluator,
    stepper,
    X: np.ndarray,
    n_steps: int = 10000,
    burn_in: int = 5000,
    thinning: int = 1,
    n_swaps_per_mala: int = 10,
    check_interval: int = 50,
) -> dict:
    """Runs a fixed-R MCMC chain: a composition kernel of
    (swap x n_swaps_per_mala) -> (MALA x 1), executed every iteration.

    Args:
        model: SparseFactorizedPrior instance (R fixed at construction)
        evaluator: PosteriorEvaluator bound to model and data X
        stepper: AdaptiveMALAStepper (File A or File B variant)
        X: observed data, shape (N, T)
        n_steps: total MCMC iterations
        burn_in: iterations discarded as burn-in
        thinning: subsampling interval for saved samples
        n_swaps_per_mala: number of swap attempts per MALA update
        check_interval: how often to adapt the MALA step size during burn-in

    Returns:
        Dict with sampled histories (z, theta, K, R, log_posterior) and
        overall accept rates.
    """
    N, T = X.shape
    S = (X @ X.T) / T

    all_pairs = model.all_pairs
    pair_probs = build_cov_bias_weights(S, all_pairs)
    pair_to_idx = {tuple(p): i for i, p in enumerate(all_pairs)}
    sigma_swap = 0.01

    samples = []
    r_history = []
    log_p_history = []

    counts = {
        "bd": {"attempts": 0, "accepts": 0},
        "mala": {"attempts": 0, "accepts": 0},
        "swap": {"attempts": 0, "accepts": 0},
    }

    recent_mala_accepts = []

    for step in range(n_steps):
        if step == burn_in:
            stepper.freeze()

        counts["bd"]["attempts"] += 1
        bd_accepted, _ = birth_death_step(model, evaluator)
        if bd_accepted:
            counts["bd"]["accepts"] += 1

        for _ in range(n_swaps_per_mala):
            counts["swap"]["attempts"] += 1
            swap_accepted, _ = swap_move_step_cov_local(
                model=model,
                evaluator=evaluator,
                pair_probs=pair_probs,
                pair_to_idx=pair_to_idx,
                sigma=sigma_swap,
            )
            if swap_accepted:
                counts["swap"]["accepts"] += 1

        counts["mala"]["attempts"] += 1
        mala_accepted, _, _ = mala_step(model, evaluator, stepper)
        if mala_accepted:
            counts["mala"]["accepts"] += 1

        recent_mala_accepts.append(1 if mala_accepted else 0)

        if step < burn_in and (step + 1) % check_interval == 0 and len(recent_mala_accepts) >= check_interval:
            recent_rate = np.mean(recent_mala_accepts[-check_interval:])
            stepper.adapt(recent_rate)

        r_history.append(model.R)
        log_p_history.append(evaluator.log_posterior())

        if step >= burn_in and (step - burn_in) % thinning == 0:
            samples.append({
                "z": model.z.copy(),
                "theta": model.theta.copy(),
                "K": model.K.copy(),
                "R": model.R,
            })

        total_attempts = sum(c["attempts"] for c in counts.values())
        total_accepts = sum(c["accepts"] for c in counts.values())

        accept_rates = {
            "bd": counts["bd"]["accepts"] / counts["bd"]["attempts"] if counts["bd"]["attempts"] > 0 else 0.0,
            "mala": counts["mala"]["accepts"] / counts["mala"]["attempts"] if counts["mala"]["attempts"] > 0 else 0.0,
            "swap": counts["swap"]["accepts"] / counts["swap"]["attempts"] if counts["swap"]["attempts"] > 0 else 0.0,
            "overall": total_accepts / total_attempts if total_attempts > 0 else 0.0,
        }

        if step % 10000 == 0:
            print(f"{step}steps---------\n")
            print(f"bd accept rate:{accept_rates["bd"]}\n")
            print(f"mala accept rate:{accept_rates['mala']}\n")
            print(f"swap accept rate:{accept_rates['swap']}\n")
            print(f"R:{model.R}\n")

    return {
        "samples": samples,
        "r_history": r_history,
        "log_p_history": log_p_history,
        "accept_rates": accept_rates,
        "final_c": stepper.c,
    }

# initialize section

In [ ]:
def initialize_from_glasso(
    N: int, R_target: int, K_glasso: np.ndarray,
    all_pairs: list[tuple[int, int]],
    eps: float = 1e-3,
    rng: np.random.Generator | None = None,
) -> "SparseFactorizedPrior":
    """Initializes z via weighted sampling WITHOUT replacement, weights
    proportional to |K_glasso_ij| -- guided by GLasso evidence, but
    stochastic across chains/seeds (required for R-hat to be meaningful;
    see module docstring). theta is seeded from K_glasso's values via the
    alpha*beta = K_glasso_ij convention (alpha=sqrt(|q|), beta=sign(q)*sqrt(|q|)).
    """
    if rng is None:
        rng = np.random.default_rng()

    weights = np.array([abs(K_glasso[i, j]) for (i, j) in all_pairs])
    weights = weights + 1e-8  # avoid zero-weight pairs being un-selectable
    probs = weights / weights.sum()

    chosen_idx = rng.choice(len(all_pairs), size=R_target, replace=False, p=probs)
    top_pairs = [all_pairs[i] for i in chosen_idx]

    model = SparseFactorizedPrior(N=N, R=0, eps=eps)
    model.z = np.array(top_pairs, dtype=int)
    theta_init = np.zeros((R_target, 2))
    for r, (i, j) in enumerate(top_pairs):
        q = K_glasso[i, j]
        theta_init[r, 0] = np.sqrt(abs(q))
        theta_init[r, 1] = np.sign(q) * np.sqrt(abs(q))
    model.set_theta(theta_init)
    return model


def initialize_random_uniform(
    N: int, R_target: int, all_pairs: list[tuple[int, int]],
    eps: float = 1e-3,
    rng: np.random.Generator | None = None,
) -> "SparseFactorizedPrior":
    """Purely random initialization, no GLasso information -- control
    condition for the convergence check (Section 7)."""
    if rng is None:
        rng = np.random.default_rng()

    chosen_idx = rng.choice(len(all_pairs), size=R_target, replace=False)
    top_pairs = [all_pairs[i] for i in chosen_idx]

    model = SparseFactorizedPrior(N=N, R=0, eps=eps)
    model.z = np.array(top_pairs, dtype=int)
    theta_init = rng.standard_normal((R_target, 2))
    model.set_theta(theta_init)
    return model

# data generation

In [ ]:
def sample_X_given_K(K: np.ndarray, T: int) -> np.ndarray:
    """Samples T iid observations x_t ~ N(0, K^{-1}), returned as (N, T)."""
    N = K.shape[0]
    cov = np.linalg.inv(K)
    X = np.random.multivariate_normal(mean=np.zeros(N), cov=cov, size=T).T
    return X


def generate_misspecified_K(
    N: int, graph_type: str = "erdos_renyi", eps: float = 1e-3
) -> np.ndarray:
    """Generates ground truth K NOT from the model's own prior, for the
    misspecified-recovery experiments (brief Section 6)."""
    if graph_type == "erdos_renyi":
        G = nx.erdos_renyi_graph(N, p=2.0 / N)
        A = nx.to_numpy_array(G)
    elif graph_type == "grid":
        side = int(np.ceil(np.sqrt(N)))
        G = nx.grid_2d_graph(side, side)
        A = nx.to_numpy_array(G)[:N, :N]
    elif graph_type == "scale_free":
        G = nx.barabasi_albert_graph(N, m=2)
        A = nx.to_numpy_array(G)
    elif graph_type == "ar1":
        rho = 0.5
        return np.array([[rho ** abs(i - j) for j in range(N)] for i in range(N)])
    else:
        raise ValueError(f"Unknown graph type: {graph_type}")

    weight_matrix = A * 0.5
    deg_sum = np.sum(np.abs(weight_matrix), axis=1)
    return weight_matrix + np.diag(deg_sum + eps)


def generate_well_specified_K(
    N: int, target_d: float = 4.0, eps: float = 1e-3
) -> tuple[np.ndarray, np.ndarray]:
    """Generates ground truth (K_true, z_true) from the model's own
    Poisson-R prior (SparseFactorizedPrior.sample_prior(), Section 1).

    Returns z_true alongside K_true (unlike generate_misspecified_K, which
    returns K only) since z_true is needed for the R-selection sanity
    check (distinct active edge count vs. true edge count).
    """
    model = SparseFactorizedPrior(N=N, R=0, eps=eps, target_d=target_d)
    K_true = model.sample_prior()
    z_true = model.z.copy()
    return K_true, z_true


def true_edge_count_from_K(K_true: np.ndarray, threshold: float = 1e-5) -> int:
    """Counts distinct active edges (upper triangle, off-diagonal) in a
    ground-truth K. Works for both well-specified and misspecified K_true."""
    N = K_true.shape[0]
    iu = np.triu_indices(N, k=1)
    return int(np.sum(np.abs(K_true[iu]) > threshold))

# select R

In [ ]:
def held_out_log_likelihood(K_hat: np.ndarray, X_test: np.ndarray) -> float:
    """log p(X_test | K_hat) up to the same normalizing constant used
    elsewhere: (T/2) logdet(K_hat) - (T/2) tr(K_hat @ S_test)."""
    T_test = X_test.shape[1]
    S_test = (X_test @ X_test.T) / T_test
    sign, logdet = np.linalg.slogdet(K_hat)
    return 0.5 * T_test * logdet - 0.5 * T_test * np.trace(K_hat @ S_test)


def select_R_by_held_out_likelihood(
    X: np.ndarray, N: int, R_grid: list[int],
    n_steps: int, burn_in: int, thinning: int,
    train_frac: float = 0.5,
    n_swaps_per_mala: int = 10,
) -> tuple[int, dict]:
    """Splits X into train/test along T, fits one chain per candidate R
    on X_train, scores each by held-out log-likelihood on X_test, and
    returns the best R.

    Returns:
        best_R: the R with highest held-out log-likelihood
        results: dict mapping R -> held-out log-likelihood
    """
    T = X.shape[1]
    T_train = int(T * train_frac)
    X_train, X_test = X[:, :T_train], X[:, T_train:]

    all_pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]
    glasso = GraphicalLassoCV().fit(X_train.T)
    K_glasso = glasso.precision_

    results = {}
    for R in R_grid:
        rng = np.random.default_rng(R)  # deterministic per-R seed for reproducibility
        model = initialize_from_glasso(N, R, K_glasso, all_pairs, rng=rng)
        evaluator = PosteriorEvaluator(model, X_train)
        stepper = AdaptiveMALAStepper()

        mcmc_res = run_mcmc(
            model=model, evaluator=evaluator, stepper=stepper,
            X=X_train, n_steps=n_steps, burn_in=burn_in, thinning=thinning,
            n_swaps_per_mala=n_swaps_per_mala,
        )

        K_hat = np.mean([s["K"] for s in mcmc_res["samples"]], axis=0)
        held_out_ll = held_out_log_likelihood(K_hat, X_test)
        results[R] = held_out_ll
        print(f"R={R:4d}  held-out log-lik={held_out_ll:.2f}")

    best_R = max(results, key=results.get)
    return best_R, results

# R_hat, ESS

In [ ]:
def compute_rhat(chains_data: np.ndarray) -> float:
    """Gelman-Rubin R-hat. chains_data: shape (n_chains, n_samples),
    a scalar functional (e.g. one K_ij entry, or log-posterior) tracked
    across chains and post-burn-in samples."""
    n_chains, n_samples = chains_data.shape
    if n_samples < 2 or n_chains < 2:
        return 1.0

    chain_means = np.mean(chains_data, axis=1)
    chain_vars = np.var(chains_data, axis=1, ddof=1)

    W = np.mean(chain_vars)
    B = n_samples * np.var(chain_means, ddof=1)

    if W == 0:
        return 1.0 if B == 0 else np.inf

    var_hat = ((n_samples - 1) / n_samples) * W + B / n_samples
    return float(np.sqrt(var_hat / W))


def compute_ess(chains_data: np.ndarray) -> float:
    """Effective sample size, pooled across chains, via the standard
    autocorrelation-sum estimator on the pooled (mean-centered per chain)
    sequence."""
    n_chains, n_samples = chains_data.shape
    total_n = n_chains * n_samples
    if n_samples < 4:
        return float(total_n)

    centered = chains_data - chains_data.mean(axis=1, keepdims=True)
    pooled = centered.flatten()

    var = np.var(pooled)
    if var == 0:
        return float(total_n)

    max_lag = min(n_samples - 1, 200)
    autocorr_sum = 0.0
    for lag in range(1, max_lag):
        c = np.mean(pooled[:-lag] * pooled[lag:]) / var
        if c < 0.05:
            break
        autocorr_sum += c

    ess = total_n / (1 + 2 * autocorr_sum)
    return float(max(1.0, ess))

# R chain_check

In [ ]:
def run_convergence_check(
    N: int = 20, T: int = 200, target_d: float = 4.0,
    n_chains_glasso: int = 2,
    n_chains_random: int = 1,
    n_steps_glasso: int = 20000,
    n_steps_random: int = 100000,
    burn_in_frac: float = 0.25,
    thinning: int = 5,
    n_swaps_per_mala: int = 10,
    graph_type: str = "sparse_factorized",
    seed: int = 0,
) -> tuple[dict, np.ndarray]:
    """Runs n_chains_glasso chains from GLasso-guided init (at the
    PRODUCTION n_steps_glasso -- matching the real experiments, so R-hat
    computed among THESE chains doubles as the actual production
    diagnostic) and n_chains_random chain(s) from random init (at a much
    larger n_steps_random).

    n_chains_random defaults to 1 (not matched to n_chains_glasso): the
    random-init group's role is NOT to have its own R-hat computed -- it
    exists purely to provide one independent, unbiased estimate of the
    true posterior, uncontaminated by GLasso's shared informational bias.
    R-hat among GLasso-init chains alone cannot detect a shared blind spot
    they all inherit from GLasso (all chains would agree with each other
    while all missing the same region) -- only comparison against a
    GLasso-independent source can catch that. A single well-run random
    chain is sufficient for that cross-check; more would mostly add cost
    without adding diagnostic power for this specific purpose.
    """
    np.random.seed(seed)
    if graph_type == "sparse_factorized":
        K_true, _ = generate_well_specified_K(N=N, target_d=target_d)
    else:
        K_true = generate_misspecified_K(N=N, graph_type=graph_type)
    X = sample_X_given_K(K_true, T=T)

    all_pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]
    glasso = GraphicalLassoCV().fit(X.T)
    K_glasso = glasso.precision_
    R_target = int(round(target_d * N / 2))

    results = {"glasso_init": [], "random_init": []}

    group_configs = [
        ("glasso_init", n_chains_glasso, n_steps_glasso,
         lambda rng: initialize_from_glasso(N, R_target, K_glasso, all_pairs, rng=rng)),
        ("random_init", n_chains_random, n_steps_random,
         lambda rng: initialize_random_uniform(N, R_target, all_pairs, rng=rng)),
    ]

    for label, n_chains, n_steps, init_fn in group_configs:
        burn_in = int(n_steps * burn_in_frac)
        print(f"\n--- {label}  (n_chains={n_chains}, n_steps={n_steps}, burn_in={burn_in}) ---")
        for c in range(n_chains):
            chain_seed = 10_000 * (0 if label == "glasso_init" else 1) + c + 1
            np.random.seed(chain_seed)
            rng = np.random.default_rng(chain_seed)

            model = init_fn(rng)
            evaluator = PosteriorEvaluator(model, X)
            stepper = AdaptiveMALAStepper()

            mcmc_res = run_mcmc(
                model=model, evaluator=evaluator, stepper=stepper, X=X,
                n_steps=n_steps, burn_in=burn_in, thinning=thinning,
                n_swaps_per_mala=n_swaps_per_mala,
            )
            z_samples = [s["z"] for s in mcmc_res["samples"]]
            edge_probs = compute_posterior_edge_probs(z_samples, N)
            results[label].append(edge_probs)
            print(f"  chain {c}: swap_accept={mcmc_res['accept_rates']['swap']:.3f}  "
                  f"mala_accept={mcmc_res['accept_rates']['mala']:.3f}  "
                  f"final R={model.R}")

    return results, K_true


def compare_convergence(results: dict) -> tuple[float, float]:
    """Compares GLasso-init vs random-init posterior edge probabilities.
    Returns (max_abs_diff, mean_abs_diff) and shows a diagonal scatter plot.
    """
    glasso_probs = np.stack(results["glasso_init"])
    random_probs = np.stack(results["random_init"])

    glasso_mean = glasso_probs.mean(axis=0)
    random_mean = random_probs.mean(axis=0)

    N = glasso_mean.shape[0]
    iu = np.triu_indices(N, k=1)
    g = glasso_mean[iu]
    r = random_mean[iu]

    max_diff = np.max(np.abs(g - r))
    mean_diff = np.mean(np.abs(g - r))
    print(f"\nMax |glasso_mean - random_mean| across all pairs: {max_diff:.4f}")
    print(f"Mean |glasso_mean - random_mean| across all pairs: {mean_diff:.4f}")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(g, r, alpha=0.6)
    ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect agreement")
    ax.set_xlabel("Edge prob (GLasso-initialized chains)")
    ax.set_ylabel("Edge prob (Randomly-initialized chains)")
    ax.set_title("Convergence check: does init matter?")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return max_diff, mean_diff

# evaluate function

In [ ]:
def compute_posterior_edge_probs(z_history: list, N: int) -> np.ndarray:
    """Posterior edge-inclusion probability matrix: for each pair (i,j),
    the fraction of pooled posterior samples in which some column selects
    that pair. z_history: list of z arrays (one per pooled sample, each
    shape (R, 2))."""
    counts = np.zeros((N, N))
    n_samples = len(z_history)
    if n_samples == 0:
        return counts

    for z in z_history:
        for (i, j) in z:
            counts[i, j] += 1
            counts[j, i] += 1

    return counts / n_samples


def relative_frobenius_error(K_est: np.ndarray, K_true: np.ndarray) -> float:
    return float(np.linalg.norm(K_est - K_true, ord="fro") / np.linalg.norm(K_true, ord="fro"))


def bf_threshold(N: int, R: float, bf: float = 3.0) -> float:
    """Posterior edge-inclusion probability threshold corresponding to a
    target Bayes factor `bf`, given N nodes and R (posterior mean active
    columns). Baseline p0 = P(a specific pair selected at random, R draws
    with replacement from M=C(N,2) candidates), matching
    SparseFactorizedPrior's sampling scheme.
    """
    M = N * (N - 1) / 2
    p0 = 1 - (1 - 1 / M) ** R
    prior_odds = p0 / (1 - p0)
    post_odds = bf * prior_odds
    return post_odds / (1 + post_odds)


def evaluate_edge_recovery(
    edge_probs: np.ndarray, K_true: np.ndarray, threshold: float = 0.5
) -> dict:
    """F1/precision/recall of the thresholded posterior edge-inclusion
    matrix against the true edge support, on the upper-triangle
    (off-diagonal) entries only."""
    from sklearn.metrics import f1_score, precision_score, recall_score

    N = K_true.shape[0]
    iu = np.triu_indices(N, k=1)

    true_labels = (np.abs(K_true[iu]) > 1e-5).astype(int)
    pred_labels = (edge_probs[iu] >= threshold).astype(int)

    return {
        "f1": f1_score(true_labels, pred_labels, zero_division=0),
        "precision": precision_score(true_labels, pred_labels, zero_division=0),
        "recall": recall_score(true_labels, pred_labels, zero_division=0),
    }


def evaluate_glasso_baseline(
    K_glasso: np.ndarray, K_true: np.ndarray, glasso_edge_threshold: float = 1e-3
) -> dict:
    """Evaluates the Graphical Lasso point estimate with the same metrics
    used for the Bayesian posterior estimate, for baseline comparison
    (brief Section 6). GLasso gives a point estimate (not probabilities),
    so its "edge_probs" equivalent is a hard 0/1 mask at
    glasso_edge_threshold.
    """
    N = K_true.shape[0]
    glasso_edge_mask = (np.abs(K_glasso) > glasso_edge_threshold).astype(float)

    recovery = evaluate_edge_recovery(glasso_edge_mask, K_true, threshold=0.5)
    frob_err = relative_frobenius_error(K_glasso, K_true)

    return {
        "glasso_f1": recovery["f1"],
        "glasso_precision": recovery["precision"],
        "glasso_recall": recovery["recall"],
        "glasso_frobenius_error": frob_err,
    }

# calibration curve

In [ ]:
def compute_calibration_curve(edge_probs_list, K_true_list, n_bins=10):
    """Pools (edge_probs, K_true) pairs across multiple seeds/replicates
    and computes the calibration curve.

    Args:
        edge_probs_list: list of (N,N) posterior edge-probability matrices
        K_true_list: list of (N,N) ground-truth precision matrices,
            same length, index-aligned with edge_probs_list
        n_bins: number of probability bins

    Returns:
        bin_centers, observed_freq, bin_counts (all np.ndarray)
    """
    all_probs = []
    all_true = []
    for edge_probs, K_true in zip(edge_probs_list, K_true_list):
        N = K_true.shape[0]
        iu = np.triu_indices(N, k=1)
        all_probs.append(edge_probs[iu])
        all_true.append((np.abs(K_true[iu]) > 1e-5).astype(int))

    probs = np.concatenate(all_probs)
    true_mask = np.concatenate(all_true)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers, observed_freq, bin_counts = [], [], []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        is_last_bin = (hi == bin_edges[-1])
        if is_last_bin:
            mask = (probs >= lo) & (probs <= hi)   # BUG FIX: inclusive upper edge
        else:
            mask = (probs >= lo) & (probs < hi)
        if mask.sum() > 0:
            bin_centers.append((lo + hi) / 2)
            observed_freq.append(true_mask[mask].mean())
            bin_counts.append(int(mask.sum()))

    return np.array(bin_centers), np.array(observed_freq), np.array(bin_counts)


def plot_calibration_curve(bin_centers, observed_freq, bin_counts,
                            title_suffix="", save_path=None):
    fig, ax = plt.subplots(figsize=(6, 6))

    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfect calibration")

    sizes = 20 + 200 * (bin_counts / bin_counts.max())
    ax.scatter(bin_centers, observed_freq, s=sizes, color="steelblue",
               edgecolor="black", zorder=3, label="observed")
    ax.plot(bin_centers, observed_freq, color="steelblue", alpha=0.5, zorder=2)

    for x, y, n in zip(bin_centers, observed_freq, bin_counts):
        ax.annotate(f"n={n}", (x, y), textcoords="offset points",
                    xytext=(0, 8), fontsize=8, ha="center")

    ax.set_xlabel("Predicted posterior edge-inclusion probability")
    ax.set_ylabel("Observed fraction of true edges")
    ax.set_title(f"Calibration curve{title_suffix}")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    return fig

# run

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="MCMC Graph Recovery Pipeline")
    parser.add_argument("--N", type=int, default=6, help="nodes")
    parser.add_argument("--T", type=int, default=200, help="numbers of X's samples")
    parser.add_argument(
        "--graph_type", type=str, default="erdos_renyi",
        choices=["erdos_renyi", "grid", "scale_free", "ar1", "sparse_factorized"],
        help="types of graphs",
    )
    parser.add_argument("--target_d", type=float, default=4.0,
                        help="target avg degree (used for R sizing and for "
                             "sparse_factorized ground truth generation)")
    parser.add_argument("--n_steps", type=int, default=20000, help="MCMC steps")
    parser.add_argument("--burn_in", type=int, default=5000, help="burn-in steps")
    parser.add_argument("--num_seeds", type=int, default=5, help="dataset replicates")
    parser.add_argument("--n_chains", type=int, default=2, help="chains per dataset")
    parser.add_argument("--save_dir", type=str, default="./results", help="save directory")
    parser.add_argument("--thinning", type=int, default=5, help="thinning")
    parser.add_argument("--n_swaps_per_mala", type=int, default=10,
                        help="swap attempts per MALA update")

    args, _ = parser.parse_known_args()
    return args


def run_multi_seed_experiment(args):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_dir = os.path.join(args.save_dir, f"{args.graph_type}_N{args.N}_{timestamp}")
    os.makedirs(exp_dir, exist_ok=True)

    print(
        f"=== start: type of graphs={args.graph_type}, N={args.N}, T={args.T},"
        f" loops={args.num_seeds}, chains={args.n_chains} ==="
    )
    print(f"save directory: {exp_dir}\n")

    all_seed_results = []
    all_edge_probs = []
    all_K_true = []

    for seed in range(args.num_seeds):
        print(f"--- [Dataset Seed {seed+1}/{args.num_seeds}] computing ---")
        np.random.seed(seed)
        if args.graph_type == "sparse_factorized":
            K_true, z_true = generate_well_specified_K(N=args.N, target_d=args.target_d)
        else:
            K_true = generate_misspecified_K(N=args.N, graph_type=args.graph_type)
        X = sample_X_given_K(K_true, T=args.T)

        # --- Graphical Lasso: used for (a) R selection, (b) chain
        # initialization, (c) baseline comparison metrics ---
        glasso = GraphicalLassoCV().fit(X.T)
        K_glasso = glasso.precision_
        all_pairs = [(i, j) for i in range(args.N) for j in range(i + 1, args.N)]

        # --- R selection via held-out likelihood (Section 6) ---
        center = args.target_d * args.N / 2
        R_grid = sorted(set(
            int(round(center * mult)) for mult in [1.2, 1.5]
            if int(round(center * mult)) > 0
        ))
        best_R, r_search_results = select_R_by_held_out_likelihood(
            X=X, N=args.N, R_grid=R_grid,
            n_steps=args.n_steps // 3, burn_in=args.burn_in // 3,
            thinning=args.thinning, n_swaps_per_mala=args.n_swaps_per_mala,
        )
        R_target = best_R
        print(f"  -> selected R={R_target} via held-out likelihood: {r_search_results}")

        # --- run mcmc for the same data, across n_chains chains ---
        chains_log_post = []
        chains_K_history = []
        chains_z_pooled = []
        chains_K_est = []
        mala_accepts, swap_accepts = [], []

        for c in range(args.n_chains):
            chain_seed = seed * 1000 + c + 1
            np.random.seed(chain_seed)
            rng = np.random.default_rng(chain_seed)  # per-chain rng (Section 4 fix)

            model = initialize_from_glasso(args.N, R_target, K_glasso, all_pairs, rng=rng)
            evaluator = PosteriorEvaluator(model, X)
            stepper = AdaptiveMALAStepper()

            mcmc_res = run_mcmc(
                model=model, evaluator=evaluator, stepper=stepper, X=X,
                n_steps=args.n_steps, burn_in=args.burn_in, thinning=args.thinning,
                n_swaps_per_mala=args.n_swaps_per_mala,
            )

            chains_log_post.append(mcmc_res["log_p_history"])

            K_samples = [s["K"] for s in mcmc_res["samples"]]
            z_samples = [s["z"] for s in mcmc_res["samples"]]

            chains_K_history.append(K_samples)
            chains_z_pooled.extend(z_samples)
            chains_K_est.append(np.mean(K_samples, axis=0))

            mala_accepts.append(mcmc_res["accept_rates"]["mala"])
            swap_accepts.append(mcmc_res["accept_rates"]["swap"])

        # --- R-hat and ESS (Log Posterior + per-entry K) ---
        log_post_array = np.array(chains_log_post)
        K_history_array = np.array(chains_K_history)  # (n_chains, n_samples, N, N)

        rhat_log_post = compute_rhat(log_post_array)
        ess_log_post = compute_ess(log_post_array)

        rhat_K_matrix = np.zeros((args.N, args.N))
        ess_K_matrix = np.zeros((args.N, args.N))
        for i in range(args.N):
            for j in range(args.N):
                rhat_K_matrix[i, j] = compute_rhat(K_history_array[:, :, i, j])
                ess_K_matrix[i, j] = compute_ess(K_history_array[:, :, i, j])

        max_rhat_K = float(np.max(rhat_K_matrix))
        min_ess_K = float(np.min(ess_K_matrix))

        # --- recovery metrics (Bayesian estimate) ---
        K_est_mean = np.mean(chains_K_est, axis=0)
        edge_probs = compute_posterior_edge_probs(chains_z_pooled, args.N)
        frob_err = relative_frobenius_error(K_est_mean, K_true)
        R_bar = float(np.mean([len(z) for z in chains_z_pooled]))
        threshold = bf_threshold(args.N, R_bar, bf=3.0)
        recovery_metrics = evaluate_edge_recovery(edge_probs, K_true, threshold=threshold)
        active_edge_count = int(np.sum(edge_probs[np.triu_indices(args.N, k=1)] > threshold))
        true_edge_count = true_edge_count_from_K(K_true)

        # --- Graphical Lasso baseline metrics (NEW, Section 8) ---
        glasso_metrics = evaluate_glasso_baseline(K_glasso, K_true)

        all_edge_probs.append(edge_probs)
        all_K_true.append(K_true)

        # --- save matrix data ---
        seed_dir = os.path.join(exp_dir, f"seed_{seed}")
        os.makedirs(seed_dir, exist_ok=True)
        np.save(os.path.join(seed_dir, "K_true.npy"), K_true)
        np.save(os.path.join(seed_dir, "K_est.npy"), K_est_mean)
        np.save(os.path.join(seed_dir, "edge_probs.npy"), edge_probs)

        seed_data = {
            "seed": seed,
            "R_target": R_target,
            "R_bar": R_bar,
            "threshold_used": threshold,
            "frobenius_error": frob_err,
            "f1": recovery_metrics["f1"],
            "precision": recovery_metrics["precision"],
            "recall": recovery_metrics["recall"],
            "active_edge_count": active_edge_count,
            "true_edge_count": true_edge_count,
            **glasso_metrics,
            "diagnostics": {
                "rhat_log_posterior": float(rhat_log_post),
                "ess_log_posterior": float(ess_log_post),
                "max_rhat_K": max_rhat_K,
                "min_ess_K": min_ess_K,
                "mala_accept_rate": float(np.mean(mala_accepts)),
                "swap_accept_rate": float(np.mean(swap_accepts)),
            },
        }
        all_seed_results.append(seed_data)

        print(f"  -> R-hat (LogPost): {rhat_log_post:.4f} | Max R-hat(K): {max_rhat_K:.4f}")
        print(f"  -> ESS (LogPost): {ess_log_post:.1f} | Min ESS(K): {min_ess_K:.1f}")
        print(f"  -> F1: {recovery_metrics['f1']:.4f} | Frobenius error: {frob_err:.4f}")
        print(f"  -> GLasso F1: {glasso_metrics['glasso_f1']:.4f} | "
              f"GLasso Frobenius error: {glasso_metrics['glasso_frobenius_error']:.4f}\n")

    # --- summary and save ---
    summary = {
        "config": vars(args),
        "timestamp": timestamp,
        "metrics_summary": {
            "frobenius_error_mean": float(np.mean([r["frobenius_error"] for r in all_seed_results])),
            "frobenius_error_std": float(np.std([r["frobenius_error"] for r in all_seed_results])),
            "f1_mean": float(np.mean([r["f1"] for r in all_seed_results])),
            "f1_std": float(np.std([r["f1"] for r in all_seed_results])),
            "glasso_f1_mean": float(np.mean([r["glasso_f1"] for r in all_seed_results])),
            "glasso_frobenius_error_mean": float(np.mean([r["glasso_frobenius_error"] for r in all_seed_results])),
            "rhat_log_post_mean": float(np.mean([r["diagnostics"]["rhat_log_posterior"] for r in all_seed_results])),
            "min_ess_K_mean": float(np.mean([r["diagnostics"]["min_ess_K"] for r in all_seed_results])),
        },
        "per_seed_results": all_seed_results,
    }

    json_path = os.path.join(exp_dir, "summary_results.json")
    with open(json_path, "w") as f:
        json.dump(summary, f, indent=4)

    # --- calibration curve (Section 9, bug-fixed), pooled across all seeds ---
    bin_centers, observed_freq, bin_counts = compute_calibration_curve(
        all_edge_probs, all_K_true, n_bins=10
    )
    plot_calibration_curve(
        bin_centers, observed_freq, bin_counts,
        title_suffix=f" ({args.graph_type}, N={args.N}, T={args.T})",
        save_path=os.path.join(exp_dir, "calibration_curve.png"),
    )

    print("=== complete ===")
    print(f"average Frobenius error: {summary['metrics_summary']['frobenius_error_mean']:.4f}")
    print(f"average GLasso Frobenius error: {summary['metrics_summary']['glasso_frobenius_error_mean']:.4f}")
    print(f"average R-hat (LogPost): {summary['metrics_summary']['rhat_log_post_mean']:.4f}")
    print(f"path: {json_path}")

    return summary

# for geweke

In [ ]:

def composed_kernel_step(model, evaluator, stepper, X, n_swaps_per_mala=10, sigma_swap=0.01):
    """Applies exactly one iteration of the same composition used inside
    run_mcmc's loop: n_swaps_per_mala swap attempts, then one MALA update
    (mala_step -- resolves to whichever variant this file defines).
    stepper must already be frozen (no step-size adaptation during the
    Geweke test -- the transition kernel must be fixed, not time-varying).
    """
    T = X.shape[1]
    S = (X @ X.T) / T
    all_pairs = model.all_pairs
    pair_probs = build_cov_bias_weights(S, all_pairs)
    pair_to_idx = {tuple(p): i for i, p in enumerate(all_pairs)}

    birth_death_step(model, evaluator)
    for _ in range(n_swaps_per_mala):
        swap_move_step_cov_local(
            model=model, evaluator=evaluator,
            pair_probs=pair_probs, pair_to_idx=pair_to_idx, sigma=sigma_swap,
        )
    mala_step(model, evaluator, stepper)


# ---------------------------------------------------------------------
# Identifiability-invariant test statistics
# ---------------------------------------------------------------------

def geweke_test_statistics(K: np.ndarray, X: np.ndarray) -> dict:
    """Test functions computed on K (and S, derived from X). All are
    invariant to the (z,theta) symmetries noted in Section 1."""
    T = X.shape[1]
    S = (X @ X.T) / T
    tr_K = np.trace(K)
    sign, logdet_K = np.linalg.slogdet(K)
    tr_KS = np.trace(K @ S)
    return {
        "trace_K": tr_K,
        "logdet_K": logdet_K,
        "trace_KS": tr_KS,
        "trace_K_squared": tr_K ** 2,
    }


# ---------------------------------------------------------------------
# Simulator 1: marginal-conditional (iid draws from the joint prior x likelihood)
# ---------------------------------------------------------------------

# --- marginal_conditional_simulator の変更 ---
def marginal_conditional_simulator(N: int, T: int, M: int, target_d: float = 3.0) -> dict:
    """R_fixedではなく、model.sample_prior()でR~Poisson(lam)から生成する版。
    birth-death込みの合成カーネルが目標とする、真の周辺事前分布に合わせる。"""
    stat_names = list(geweke_test_statistics(np.eye(N), np.ones((N, 1))).keys())
    results = {name: [] for name in stat_names}

    for _ in range(M):
        model = SparseFactorizedPrior(N=N, target_d=target_d)  # R=0で初期化
        model.sample_prior()  # ここでR~Poisson(lam)から(z,theta)を生成、model.Kも更新
        X = sample_X_given_K(model.K, T=T)
        stats = geweke_test_statistics(model.K, X)
        for name in stat_names:
            results[name].append(stats[name])

    return {name: np.array(vals) for name, vals in results.items()}


# --- successive_conditional_simulator の変更 ---
def successive_conditional_simulator(
    N: int, T: int, M: int, n_swaps_per_mala: int = 10, target_d: float = 3.0,
) -> dict:
    stat_names = list(geweke_test_statistics(np.eye(N), np.ones((N, 1))).keys())
    results = {name: [] for name in stat_names}

    model = SparseFactorizedPrior(N=N, target_d=target_d)
    model.sample_prior()  # 初期状態もPoisson事前分布から始める
    stepper = AdaptiveMALAStepper()
    stepper.freeze()

    for _ in range(M):
        X = sample_X_given_K(model.K, T=T)
        evaluator = PosteriorEvaluator(model, X)
        composed_kernel_step(model, evaluator, stepper, X, n_swaps_per_mala=n_swaps_per_mala)
        stats = geweke_test_statistics(model.K, X)
        for name in stat_names:
            results[name].append(stats[name])

    return {name: np.array(vals) for name, vals in results.items()}


# ---------------------------------------------------------------------
# Comparison: Z-scores with Bonferroni correction
# ---------------------------------------------------------------------

def run_geweke_test(
    N: int = 6, R_fixed: int = 6, T: int = 50, M: int = 2000,
    n_swaps_per_mala: int = 10, eps: float = 1e-3,
) -> dict:
    """Runs both simulators and compares each test statistic via a
    two-sample Z-score. Uses a Bonferroni-corrected threshold (alpha=0.05
    / n_tests) rather than the raw 1.96 cutoff, since 4 statistics are
    tested jointly.
    """
    print(f"Running marginal-conditional simulator (M={M})...")
    mc_results = marginal_conditional_simulator(N, R_fixed, T, M)

    print(f"Running successive-conditional simulator (M={M})...")
    sc_results = successive_conditional_simulator(N, R_fixed, T, M, n_swaps_per_mala)

    # --- Point 2: closed-form sanity check on the MC side ---
    expected_trace_K = N * eps + 2 * R_fixed
    mc_trace_K_mean = mc_results["trace_K"].mean()
    print(f"\n[Point 2 check] E[trace(K)] closed-form = {expected_trace_K:.4f}  "
          f"|  MC empirical mean = {mc_trace_K_mean:.4f}  "
          f"|  diff = {abs(expected_trace_K - mc_trace_K_mean):.4f}")
    if abs(expected_trace_K - mc_trace_K_mean) > 0.1 * expected_trace_K:
        print("  !! WARNING: MC mean deviates >10% from closed form -- "
              "check the harness/prior simulator before trusting SC comparisons.")

    stat_names = list(mc_results.keys())
    n_tests = len(stat_names)
    alpha_bonferroni = 0.05 / n_tests
    from scipy.stats import norm
    z_critical = norm.ppf(1 - alpha_bonferroni / 2)  # two-sided

    verdict = {}
    print(f"{'Statistic':<20} {'MC mean':>10} {'SC mean':>10} {'ESS(SC)':>9} {'Z-score':>10} {'verdict':>10}")
    print("-" * 50)
    for name in stat_names:
        mc_vals = mc_results[name]
        sc_vals = sc_results[name]

        mean_diff = mc_vals.mean() - sc_vals.mean()
        ess_sc = compute_ess(sc_vals.reshape(1, -1))
        se = np.sqrt(mc_vals.var(ddof=1) / M + sc_vals.var(ddof=1) / ess_sc)
        z = mean_diff / se if se > 0 else np.nan

        reject = bool(np.abs(z) > z_critical)
        verdict[name] = {"z_score": float(z), "reject": reject,
                  "mc_mean": float(mc_vals.mean()), "sc_mean": float(sc_vals.mean()),
                  "ess_sc": float(ess_sc)}
        flag = "REJECT" if reject else "pass"
        print(f"{name:<20} {mc_vals.mean():>10.4f} {sc_vals.mean():>10.4f} {ess_sc:>9.1f} {z:>10.3f} {flag:>10}")

    overall_pass = not any(v["reject"] for v in verdict.values())
    print(f"\nOverall: {'PASS' if overall_pass else 'FAIL -- investigate before trusting the sampler'}")

    return {"verdict": verdict, "overall_pass": overall_pass,
            "mc_results": mc_results, "sc_results": sc_results}




# If this passes, consider re-running at a size closer to the actual
# experiments (e.g. N=20) as a final confirmation before trusting the
# full N=20 recovery/calibration results.

In [ ]:
def unit_test_update_factor(
    N: int = 10, R_fixed: int = 8, T: int = 20, n_trials: int = 3000,
    seed: int = 0,
) -> dict:
    """advisor検定1: update_factor(rank-1 Cholesky更新)が、
    ゼロから全部計算し直した'愚直な'結果と機械精度で一致するかを確認する。
    swap_move_step_cov_local自体は呼ばず、update_factorのみを対象にする。
    """
    rng = np.random.default_rng(seed)
    all_pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]

    K_diffs = []       # K自体の誤差
    L_diffs = []       # Cholesky因子Lの誤差
    lik_diffs = []      # そこから計算されるlog_likelihoodの誤差

    n_valid = 0
    for trial in range(n_trials):
        # 1. 適当な(z,theta)を事前分布から用意
        model = SparseFactorizedPrior(N=N, R=R_fixed)
        X = rng.standard_normal((N, T))  # モデル由来である必要はない
        evaluator = PosteriorEvaluator(model, X)

        # 2. ランダムなswap提案の中身だけを作る(受理判定はしない)
        r = rng.integers(R_fixed)
        new_pair_idx = rng.integers(len(all_pairs))
        new_pair = tuple(all_pairs[new_pair_idx])
        theta_new_r = rng.standard_normal(2)

        # --- 近道: update_factor を直接呼ぶ ---
        success = model.update_factor(r, new_pair, theta_new_r)
        if not success:
            continue  # PD性が崩れるケースはこの検定の対象外
        n_valid += 1

        K_fast = model.K.copy()
        L_fast = model.L.copy()
        loglik_fast = evaluator.log_likelihood()

        # --- 愚直: Kをゼロから作り直し、フルCholesky分解 ---

        B_slow = np.zeros((N, R_fixed))
        for i in range(R_fixed):
            ii, jj = model.z[i]
            alpha, beta = model.theta[i]
            B_slow[ii, i] = alpha
            B_slow[jj, i] = beta
        K_slow = model.eps * np.eye(N) + B_slow @ B_slow.T
        K_slow = model.eps * np.eye(N) + B_slow @ B_slow.T
        L_slow = np.linalg.cholesky(K_slow)

        S = evaluator.S
        T_ = evaluator.T
        logdet_slow = 2.0 * np.sum(np.log(np.diag(L_slow)))
        loglik_slow = 0.5 * T_ * logdet_slow - 0.5 * T_ * np.trace(K_slow @ S)

        # --- 比較 ---
        K_diffs.append(np.max(np.abs(K_fast - K_slow)))
        L_diffs.append(np.max(np.abs(L_fast - L_slow)))
        lik_diffs.append(abs(loglik_fast - loglik_slow))

    K_diffs = np.array(K_diffs)
    L_diffs = np.array(L_diffs)
    lik_diffs = np.array(lik_diffs)

    print(f"有効試行数: {n_valid}/{n_trials}")
    print(f"K の誤差:        最大={K_diffs.max():.2e}, 平均={K_diffs.mean():.2e}")
    print(f"L(Cholesky)の誤差: 最大={L_diffs.max():.2e}, 平均={L_diffs.mean():.2e}")
    print(f"log_likelihoodの誤差: 最大={lik_diffs.max():.2e}, 平均={lik_diffs.mean():.2e}")

    if lik_diffs.max() > 1e-8:
        print("!! WARNING: update_factorに、機械精度を超える誤差があります。バグの可能性が高いです。")
    else:
        print("OK: update_factorは機械精度で正しく動作しています。")

    return {"K_diffs": K_diffs, "L_diffs": L_diffs, "lik_diffs": lik_diffs}

In [ ]:
result = unit_test_update_factor(N=10, R_fixed=8, T=20, n_trials=3000)

有効試行数: 3000/3000
K の誤差:        最大=3.55e-15, 平均=3.59e-16
L(Cholesky)の誤差: 最大=4.40e-13, 平均=4.84e-15
log_likelihoodの誤差: 最大=5.02e-11, 平均=9.52e-13
OK: update_factorは機械精度で正しく動作しています。


In [ ]:
def plot_geweke_qq(mc_results: dict, sc_results: dict, stat_name: str,
                    save_path: str = None):
    """QQ plot comparing the marginal-conditional (MC) and successive-
    conditional (SC) distributions of a single test statistic.
    """
    import matplotlib.pyplot as plt

    mc_vals = np.sort(mc_results[stat_name])
    sc_vals = np.sort(sc_results[stat_name])

    n_points = min(len(mc_vals), len(sc_vals), 2000)
    quantiles = np.linspace(0, 1, n_points)
    mc_q = np.quantile(mc_vals, quantiles)
    sc_q = np.quantile(sc_vals, quantiles)

    fig, ax = plt.subplots(figsize=(6, 6))
    lo = min(mc_q.min(), sc_q.min())
    hi = max(mc_q.max(), sc_q.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="gray", label="y = x (perfect match)")
    ax.scatter(mc_q, sc_q, s=8, alpha=0.6, color="steelblue")

    ax.set_xlabel(f"MC quantiles ({stat_name})")
    ax.set_ylabel(f"SC quantiles ({stat_name})")
    ax.set_title(f"Q-Q plot: {stat_name}")
    ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    return fig

plot_geweke_qq(result["mc_results"], result["sc_results"], "logdet_K")

KeyError: 'mc_results'

In [ ]:
def count_duplicate_pairs(z: np.ndarray) -> int:
    """z: (R,2) の支持点配列。重複しているペアの本数を数える
    (例: R=6本のうち2本が同じペアなら、重複数=1)。
    """
    pairs = [tuple(p) for p in z]
    return len(pairs) - len(set(pairs))


def check_duplicate_bias(N: int = 6, R_fixed: int = 6, T: int = 50, M: int = 2000,
                          n_swaps_per_mala: int = 10) -> dict:
    """MC(事前分布) と SC(MCMC) それぞれで、重複ペア数の分布を比較する
    軽量診断。logdet_K のQQプロットで見られた分布の圧縮が、
    swapが重複の少ない状態ばかり好む偏りで説明できるかを直接確認する。
    """
    mc_dup_counts = []
    for _ in range(M):
        model = SparseFactorizedPrior(N=N, R=R_fixed)
        mc_dup_counts.append(count_duplicate_pairs(model.z))
    mc_dup_counts = np.array(mc_dup_counts)

    sc_dup_counts = []
    model = SparseFactorizedPrior(N=N, R=R_fixed)
    stepper = AdaptiveMALAStepper()
    stepper.freeze()
    for _ in range(M):
        X = sample_X_given_K(model.K, T=T)
        evaluator = PosteriorEvaluator(model, X)
        composed_kernel_step(model, evaluator, stepper, X, n_swaps_per_mala=n_swaps_per_mala)
        sc_dup_counts.append(count_duplicate_pairs(model.z))
    sc_dup_counts = np.array(sc_dup_counts)

    print(f"MC: 重複本数の平均={mc_dup_counts.mean():.3f}, 分布={np.bincount(mc_dup_counts)}")
    print(f"SC: 重複本数の平均={sc_dup_counts.mean():.3f}, 分布={np.bincount(sc_dup_counts)}")

    return {"mc_dup_counts": mc_dup_counts, "sc_dup_counts": sc_dup_counts}

In [ ]:
# result は M=500000 のときの run_geweke_test の戻り値と仮定
# result["sc_results"] に各統計量の時系列があるが、重複数自体は記録されていないため、
# 別途 successive_conditional_simulator を M=500000 で回して z の履歴から計算する必要がある。
# もし既に sc_dup_counts のような配列を持っていればそれを使う。

# 仮に、既存の check_duplicate_bias 系の結果が M=500000 でまだ無ければ、以下で計算:
dup_result_500k = check_duplicate_bias(N=6, R_fixed=6, T=50, M=500000, n_swaps_per_mala=10)

dup_free_count = dup_result_500k["sc_dup_counts"] == 0
dup_free_rate = dup_free_count.mean()
print(f"M=500000での重複ゼロ率: {dup_free_rate:.4f} (理論値: 0.3080)")

In [ ]:
def check_composed_multi_chain(
    N: int = 6, R_fixed: int = 6, T: int = 5, n_chains: int = 30,
    n_swaps_per_mala: int = 10, n_steps_per_chain: int = 2000,
    seed: int = 0,
) -> dict:
    """advisor検定2: 事前分布から新規drawした30本の独立チェーンを走らせ、
    chainごとの平均値を使い、chain間の分散を標準誤差として使う。
    ESS推定に依存しないため、その問題の影響を受けない。
    logdet(K)と重複ゼロ率、両方を検証する。
    """
    rng_master = np.random.default_rng(seed)

    # --- MC側: 理論値との比較用に、同じ規模でまっすぐ生成 ---
    mc_logdet = []
    mc_dupfree = []
    M_mc = n_chains * n_steps_per_chain  # SC側と同程度のサンプル数に揃える
    for _ in range(M_mc):
        model = SparseFactorizedPrior(N=N, R=R_fixed)
        sign, logdet = np.linalg.slogdet(model.K)
        mc_logdet.append(logdet)
        mc_dupfree.append(count_duplicate_pairs(model.z) == 0)
    mc_logdet = np.array(mc_logdet)
    mc_dupfree_rate = np.mean(mc_dupfree)

    # --- SC側: 30本の独立チェーン、それぞれ事前分布から新規draw ---
    chain_means_logdet = []
    chain_dupfree_rates = []

    for c in range(n_chains):
        chain_seed = seed * 1000 + c + 1
        np.random.seed(chain_seed)
        model = SparseFactorizedPrior(N=N, R=R_fixed)  # 新規draw
        stepper = AdaptiveMALAStepper()
        stepper.freeze()

        logdet_history = []
        dupfree_history = []
        for _ in range(n_steps_per_chain):
            X = sample_X_given_K(model.K, T=T)
            evaluator = PosteriorEvaluator(model, X)
            composed_kernel_step(model, evaluator, stepper, X, n_swaps_per_mala=n_swaps_per_mala)

            sign, logdet = np.linalg.slogdet(model.K)
            logdet_history.append(logdet)
            dupfree_history.append(count_duplicate_pairs(model.z) == 0)

        chain_means_logdet.append(np.mean(logdet_history))
        chain_dupfree_rates.append(np.mean(dupfree_history))

    chain_means_logdet = np.array(chain_means_logdet)
    chain_dupfree_rates = np.array(chain_dupfree_rates)

    # --- Z検定: chain間の分散を標準誤差として使う ---
    mc_mean = mc_logdet.mean()
    sc_grand_mean = chain_means_logdet.mean()
    se_between_chain = chain_means_logdet.std(ddof=1) / np.sqrt(n_chains)
    # MC側の誤差も考慮(M_mcが十分大きいので、通常SC側の誤差が支配的)
    se_mc = mc_logdet.std(ddof=1) / np.sqrt(M_mc)
    se_total = np.sqrt(se_between_chain**2 + se_mc**2)
    z = (mc_mean - sc_grand_mean) / se_total

    print(f"[logdet(K)]")
    print(f"  MC mean = {mc_mean:.4f}")
    print(f"  SC grand mean (30 chains) = {sc_grand_mean:.4f}")
    print(f"  between-chain SE = {se_between_chain:.4f}")
    print(f"  Z = {z:.3f}")
    print()
    print(f"[重複ゼロ率]")
    print(f"  MC = {mc_dupfree_rate:.4f} (理論値0.3080)")
    print(f"  SC grand mean (30 chains) = {chain_dupfree_rates.mean():.4f}")
    print(f"  SC per-chain std = {chain_dupfree_rates.std(ddof=1):.4f}")

    return {
        "chain_means_logdet": chain_means_logdet,
        "chain_dupfree_rates": chain_dupfree_rates,
        "z_logdet": z,
        "mc_dupfree_rate": mc_dupfree_rate,
    }

In [ ]:
def check_composed_multi_chain_all_stats(
    N: int = 6, T: int = 5, n_chains: int = 30,
    n_swaps_per_mala: int = 10, n_steps_per_chain: int = 2000,
    seed: int = 0, target_d: float = 3.0,
) -> dict:
    """... (docstring)
    R_fixedではなく、sample_prior()でR~Poisson(lam)から生成する版。
    birth-death込みの合成カーネルが目標とする、真の周辺事前分布に合わせる。
    """
    stat_names = list(geweke_test_statistics(np.eye(N), np.ones((N, 1))).keys())

    # --- MC側 ---
    mc_stats = {name: [] for name in stat_names}
    mc_dupfree = []
    M_mc = n_chains * n_steps_per_chain
    for _ in range(M_mc):
        model = SparseFactorizedPrior(N=N, target_d=target_d)
        model.sample_prior()          # ← 変更: R~Poisson(lam)から生成
        X = sample_X_given_K(model.K, T=T)
        stats = geweke_test_statistics(model.K, X)
        for name in stat_names:
            mc_stats[name].append(stats[name])
        mc_dupfree.append(count_duplicate_pairs(model.z) == 0)
    mc_stats = {name: np.array(vals) for name, vals in mc_stats.items()}
    mc_dupfree_rate = np.mean(mc_dupfree)

    # --- SC側 ---
    chain_means = {name: [] for name in stat_names}
    chain_dupfree_rates = []

    for c in range(n_chains):
        chain_seed = seed * 1000 + c + 1
        np.random.seed(chain_seed)
        model = SparseFactorizedPrior(N=N, target_d=target_d)
        model.sample_prior()          # ← 変更: 初期状態もPoisson事前分布から
        stepper = AdaptiveMALAStepper()
        stepper.freeze()

        stat_history = {name: [] for name in stat_names}
        dupfree_history = []
        for _ in range(n_steps_per_chain):
            X = sample_X_given_K(model.K, T=T)
            evaluator = PosteriorEvaluator(model, X)
            composed_kernel_step(model, evaluator, stepper, X, n_swaps_per_mala=n_swaps_per_mala)

            stats = geweke_test_statistics(model.K, X)
            for name in stat_names:
                stat_history[name].append(stats[name])
            dupfree_history.append(count_duplicate_pairs(model.z) == 0)

        for name in stat_names:
            chain_means[name].append(np.mean(stat_history[name]))
        chain_dupfree_rates.append(np.mean(dupfree_history))

    chain_means = {name: np.array(vals) for name, vals in chain_means.items()}
    chain_dupfree_rates = np.array(chain_dupfree_rates)

    # --- Z検定 → t検定に変更 (df = n_chains - 1) ---
    from scipy.stats import t as t_dist
    alpha_bonferroni = 0.05 / len(stat_names)
    df = n_chains - 1
    t_critical = t_dist.ppf(1 - alpha_bonferroni / 2, df=df)   # ← 変更

    verdict = {}
    print(f"{'Statistic':<18} {'MC mean':>12} {'SC mean':>12} {'t-score':>10} {'verdict':>10}")
    print("-" * 64)
    for name in stat_names:
        mc_vals = mc_stats[name]
        sc_chain_means = chain_means[name]

        mc_mean = mc_vals.mean()
        sc_grand_mean = sc_chain_means.mean()
        se_between_chain = sc_chain_means.std(ddof=1) / np.sqrt(n_chains)
        se_mc = mc_vals.std(ddof=1) / np.sqrt(M_mc)
        se_total = np.sqrt(se_between_chain**2 + se_mc**2)
        t_stat = (mc_mean - sc_grand_mean) / se_total if se_total > 0 else np.nan   # ← 変数名変更

        reject = bool(np.abs(t_stat) > t_critical)   # ← 変更
        verdict[name] = {
            "t_score": float(t_stat), "reject": reject,
            "mc_mean": float(mc_mean), "sc_grand_mean": float(sc_grand_mean),
            "se_between_chain": float(se_between_chain),
        }
        flag = "REJECT" if reject else "pass"
        print(f"{name:<18} {mc_mean:>12.4f} {sc_grand_mean:>12.4f} {t_stat:>10.3f} {flag:>10}")

    overall_pass = not any(v["reject"] for v in verdict.values())
    print(f"\nOverall: {'PASS' if overall_pass else 'FAIL -- investigate before trusting the sampler'}")

    print(f"\n[dup-free rate]")
    print(f"  MC = {mc_dupfree_rate:.4f}")
    print(f"  SC grand mean ({n_chains} chains) = {chain_dupfree_rates.mean():.4f}")
    print(f"  SC per-chain std = {chain_dupfree_rates.std(ddof=1):.4f}")

    return {
        "verdict": verdict,
        "overall_pass": overall_pass,
        "chain_means": chain_means,
        "mc_stats": mc_stats,
        "chain_dupfree_rates": chain_dupfree_rates,
        "mc_dupfree_rate": mc_dupfree_rate,
    }

In [ ]:
result = check_composed_multi_chain_all_stats(
    N=6, T=5, n_chains=30, n_swaps_per_mala=10,
    n_steps_per_chain=20000, seed=0,
)

Statistic               MC mean      SC mean    t-score    verdict
----------------------------------------------------------------
trace_K                 17.9985      18.4166     -1.688       pass
logdet_K                -1.2755      -0.9291     -1.522       pass
trace_KS                 5.9986       6.0033     -1.733       pass
trace_K_squared        395.9396     410.2487     -1.444       pass

Overall: PASS

[dup-free rate]
  MC = 0.1424
  SC grand mean (30 chains) = 0.1375
  SC per-chain std = 0.0334


In [ ]:
result_multi_chain = check_composed_multi_chain(
    N=6, R_fixed=6, T=5, n_chains=30, n_swaps_per_mala=10, n_steps_per_chain=2000
)

In [ ]:
dup_result = check_duplicate_bias(N=6, R_fixed=6, T=50, M=50000, n_swaps_per_mala=10)

In [ ]:
ess_sc = compute_ess(dup_result["sc_dup_counts"].reshape(1, -1))
print(f"ESS(SC) = {ess_sc:.1f}")

import numpy as np
p_mc = 0.31332  # MC側の実測値(ほぼ理論値と一致)
p_sc = dup_result["sc_dup_counts"] == 0
p_sc_mean = p_sc.mean()

se_sc = np.sqrt(p_sc_mean * (1 - p_sc_mean) / ess_sc)
z = (p_mc - p_sc_mean) / se_sc
print(f"SC重複ゼロ割合 = {p_sc_mean:.4f}, SE = {se_sc:.4f}, Z = {z:.3f}")

In [ ]:
def swap_move_uniform_pair(model, evaluator, all_pairs, sigma: float = 0.01):
    """(a) 一様なペア提案 + ランダムウォークのtheta。"""
    R = model.R
    if R == 0:
        return False, -np.inf

    z_old = model.z.copy()
    theta_old = model.theta.copy()
    K_old, L_old = model.K.copy(), model.L.copy()

    log_lik_before = evaluator.log_likelihood()

    r = np.random.randint(R)
    old_pair = tuple(model.z[r])

    new_pair_idx = np.random.randint(len(all_pairs))
    new_pair = tuple(all_pairs[new_pair_idx])

    theta_old_r = model.theta[r].copy()
    theta_new_r = theta_old_r + sigma * np.random.randn(2)

    success = model.update_factor(r, new_pair, theta_new_r)
    if not success:
        model.z, model.theta, model.K, model.L = z_old, theta_old, K_old, L_old
        model._rebuild_B()
        return False, -np.inf

    log_lik_after = evaluator.log_likelihood()

    log_prior_ratio = -0.5 * (np.sum(theta_new_r**2) - np.sum(theta_old_r**2))
    log_ratio = (log_lik_after - log_lik_before) + log_prior_ratio

    if np.log(np.random.rand()) < log_ratio:
        return True, log_ratio
    else:
        model.z, model.theta, model.K, model.L = z_old, theta_old, K_old, L_old
        model._rebuild_B()
        return False, log_ratio

def check_duplicate_bias_variant_a(
    N: int = 6, R_fixed: int = 6, T: int = 50, M: int = 2000,
    n_swaps_per_mala: int = 10,
) -> dict:
    """一様ペア提案版での重複バイアス検定 + 受理率の計測"""
    all_pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]

    mc_dup_counts = []
    for _ in range(M):
        model = SparseFactorizedPrior(N=N, R=R_fixed)
        mc_dup_counts.append(count_duplicate_pairs(model.z))
    mc_dup_counts = np.array(mc_dup_counts)

    sc_dup_counts = []
    n_accepts = 0
    n_attempts = 0
    model = SparseFactorizedPrior(N=N, R=R_fixed)
    for _ in range(M):
        X = sample_X_given_K(model.K, T=T)
        evaluator = PosteriorEvaluator(model, X)
        for _ in range(n_swaps_per_mala):
            accepted, _ = swap_move_uniform_pair(model, evaluator, all_pairs, sigma=0.01)
            n_attempts += 1
            if accepted:
                n_accepts += 1
        sc_dup_counts.append(count_duplicate_pairs(model.z))
    sc_dup_counts = np.array(sc_dup_counts)

    accept_rate = n_accepts / n_attempts
    print(f"受理率: {accept_rate:.4f}  ({n_accepts}/{n_attempts})")
    print(f"MC: 重複本数の平均={mc_dup_counts.mean():.3f}, 分布={np.bincount(mc_dup_counts)}")
    print(f"SC: 重複本数の平均={sc_dup_counts.mean():.3f}, 分布={np.bincount(sc_dup_counts)}")

    return {"mc_dup_counts": mc_dup_counts, "sc_dup_counts": sc_dup_counts,
            "accept_rate": accept_rate}